In [2]:
%pip install aiohttp requests

from shapely.geometry import Polygon
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os

fiona.supported_drivers["KML"] = "rw"  # Enable kml driver

Note: you may need to restart the kernel to use updated packages.


In [3]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

In [4]:
# find all Biomass collections
results = catalog.collection_search(
    filter="platform='Biomass'",
    datetime="2025-12-01T00:00:00.000Z/..",
)
print(f"{results.matched()} collections found.")

data = results.collection_list_as_dict()
df = pd.json_normalize(data, record_path=["collections"]).sort_values(
    by="title", ignore_index=True
)
df[["id", "title", "extent.temporal.interval"]]

17 collections found.


,id,title,extent.temporal.interval
0,BiomassAux,Biomass Auxiliary,"[[2024-07-31T00:00:00.000Z, None]]"
1,BiomassAuxIOC,Biomass Auxiliary (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
2,BiomassAuxRest,Biomass Auxiliary Restricted,"[[2024-07-31T00:00:00.000Z, None]]"
3,BiomassCalVal10,Biomass Cal/Val,"[[2025-04-29T00:00:00.000Z, None]]"
4,BiomassLevel0,Biomass Level 0,"[[2024-07-31T00:00:00.000Z, None]]"
5,BiomassLevel0IOC,Biomass Level 0 (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
6,BiomassLevel1a,Biomass Level 1A,"[[2025-11-21T00:00:00.000Z, None]]"
7,BiomassLevel1aIOC,Biomass Level 1A (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
8,BiomassLevel1b,Biomass Level 1B,"[[2024-07-31T00:00:00.000Z, None]]"
9,BiomassLevel1bIOC,Biomass Level 1B (IOC),"[[2024-07-31T00:00:00.000Z, None]]"


In [5]:
search = catalog.search(
    collections = ["BiomassLevel2a"],
    filter = "product:type='FP_FH__L2A'", 
    datetime = ['2026-01-01T00:00:00Z','2026-07-01T23:59:59Z'],
    bbox = [0.55802, -0.31476, -13.02363, -0.31476], # Parc national d'Ivindo
    method = "GET",
    max_items = 10,  
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

Accessing 10 items (limited by max_items).
95 items found that matched the query.
